<a href="https://colab.research.google.com/github/nieblasIX/IX-SierraGorda-Monitoring/blob/main/01_SierraGorda_Degradacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Instalamos la herramienta para ver mapas (geemap)
!pip install geemap

# 2. Llamamos a Google Earth Engine
import ee
import geemap

# 3. Pedimos permiso para usar tu cuenta (Autenticación)
ee.Authenticate()
ee.Initialize()
# OJO: Si no tienes un proyecto de Google Cloud, borra lo de adentro del paréntesis y deja solo: ee.Initialize()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.2 MB/s eta 0:00:00


EEException: ee.Initialize: no project found. Call with project= or see http://goo.gle/ee-auth.

In [14]:
# Reemplaza 'TU-PROYECTO-AQUI' con el ID real de tu proyecto de Google Cloud
proyecto_id = 'sierragorda-2026'

ee.Initialize(project=proyecto_id)

# Creamos el mapa interactivo
Map = geemap.Map()

# Nombres de los municipios
#municipios = ['Pisaflores', 'La Mision', 'Jacala De Ledezma', 'Pacula', 'Zimapan']
municipios = ['La Mision']

# Buscamos México y filtramos la Sierra Gorda
mexico = ee.FeatureCollection("FAO/GAUL/2015/level2")
sierra_gorda = mexico.filter(ee.Filter.inList('ADM2_NAME', municipios))

# Centramos el mapa y lo pintamos de rojo
Map.centerObject(sierra_gorda, 10)
Map.addLayer(sierra_gorda, {'color': 'red'}, 'Sierra Gorda Hidalguense')

# Mostramos el mapa
Map

Map(center=[21.093185392396755, -99.06433549097217], controls=(WidgetControl(options=['position', 'transparent…

In [15]:
# Usamos satélites Landsat 8 de los últimos años
l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(sierra_gorda) \
    .filterDate('2018-01-01', '2023-12-31')

# Función matemática (Índice NBR: detecta estrés hídrico y biomasa)
def calcular_salud(image):
    nbr = image.normalizedDifference(['SR_B5', 'SR_B7']).rename('Salud_Forestal')
    return image.addBands(nbr)

# Aplicamos la función y sacamos el promedio
l8_salud = l8.map(calcular_salud)
promedio_salud = l8_salud.select('Salud_Forestal').mean().clip(sierra_gorda)

# NUEVA PALETA DE ALTO CONTRASTE (Cartografía Profesional)
# Rojo oscuro -> Naranja -> Amarillo -> Verde claro -> Verde esmeralda
colores_salud = {
    'min': -0.1,
    'max': 0.6,
    'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']
}

# Añadimos al mapa
Map.addLayer(promedio_salud, colores_salud, 'Salud Forestal (Alto Contraste)')
Map

Map(bottom=115653.0, center=[21.093185392396755, -99.06433549097217], controls=(WidgetControl(options=['positi…

In [16]:
# 1. VARIABLES PREDICTORAS (Lo que la IA va a estudiar)
# Añadimos un Modelo de Elevación Digital (Topografía) a nuestro Landsat
srtm = ee.Image("USGS/SRTMGL1_003").clip(sierra_gorda)
predictores = promedio_salud.addBands(srtm).addBands(l8.mean()) # NBR + Altitud + Bandas de luz
bandas_entrenamiento = predictores.bandNames()

# 2. DATOS DE VERDAD TERRENO (Nuestro "Maestro")
# Usamos ESA WorldCover (10=Bosque, 20=Matorral, 30=Pasto, 40=Agricultura)
esa_lc = ee.ImageCollection("ESA/WorldCover/v100").first().clip(sierra_gorda)

# 3. MUESTREO ALEATORIO ESTRATIFICADO
# Tiramos 1500 puntos al azar en la Sierra Gorda para extraer la "firma espectral"
puntos_entrenamiento = esa_lc.addBands(predictores).stratifiedSample(
    numPoints=300, # 300 puntos por cada clase de vegetación
    classBand='Map',
    region=sierra_gorda,
    scale=30,
    geometries=True
)

print("Puntos de entrenamiento generados con éxito. ¡El set de datos está listo!")

Puntos de entrenamiento generados con éxito. ¡El set de datos está listo!


In [17]:
# --- ALGORITMO 1: CART (Rápido y sencillo) ---
clasificador_cart = ee.Classifier.smileCart().train(
    features=puntos_entrenamiento,
    classProperty='Map',
    inputProperties=bandas_entrenamiento
)
mapa_cart = predictores.classify(clasificador_cart)

# --- ALGORITMO 2: RANDOM FOREST (El estándar de oro) ---
# Bajamos los árboles a 30 para ahorrar memoria sin perder mucha precisión
clasificador_rf = ee.Classifier.smileRandomForest(numberOfTrees=30).train(
    features=puntos_entrenamiento,
    classProperty='Map',
    inputProperties=bandas_entrenamiento
)
mapa_rf = predictores.classify(clasificador_rf)

print("Modelos CART y Random Forest entrenados. SVM descartado por exceso de memoria.")

Modelos CART y Random Forest entrenados. SVM descartado por exceso de memoria.


In [18]:
# Paleta de colores (ESA WorldCover)
# 10: Bosque (Verde Oscuro), 20: Matorral (Naranja), 30: Pasto/Agricultura (Amarillo)
paleta_vegetacion = {
    'min': 10, 'max': 40,
    'palette': ['#006400', '#ffbb22', '#ffff4c', '#f096ff']
}

# Creamos el mapa
Map2 = geemap.Map()
Map2.centerObject(sierra_gorda, 10)

# Resample the classified maps to a coarser scale for visualization to reduce memory
# The original data might be at 30m, we'll reproject to 5000m for display
display_scale = 5000 # meters

mapa_cart_display = mapa_cart.reproject(crs=mapa_cart.projection().crs(), scale=display_scale)
mapa_rf_display = mapa_rf.reproject(crs=mapa_rf.projection().crs(), scale=display_scale)

# Añadimos los mapas (CART empieza apagado para no saturar la memoria)
Map2.addLayer(mapa_cart_display, paleta_vegetacion, '1. Clasificación CART (Sencilla)', False)
Map2.addLayer(mapa_rf_display, paleta_vegetacion, '2. Clasificación Random Forest (Óptima)', True)

# Añadimos los límites
Map2.addLayer(ee.Image().paint(sierra_gorda, 0, 2), {'palette': 'black'}, 'Límites Municipales')

Map2

Map(center=[21.093185392396784, -99.06433549097163], controls=(WidgetControl(options=['position', 'transparent…